In [1]:
from Data_query.trino_config import *
import numpy as np
import boto3
session = boto3.Session()
s3 = boto3.client('s3')

In [6]:
stop_trino()


Trino service stopping triggered.


In [2]:
ensure_trino_running(worker_desired_count = 0, big_worker_desired_count=4)
sleep(30)


Trino service is not running. Starting the service...
Trino service triggered.
Service trino-service is now stable.


In [3]:
trino_exec("DROP TABLE IF EXISTS conformance_voltwatt")
trino_exec("""CREATE TABLE conformance_voltwatt (
                year BIGINT,
                month BIGINT,
                day BIGINT,
                site_id BIGINT,
                nonconformance_voltwatt_sum DOUBLE,
                nonconformance_voltwatt_count BIGINT,
                total_count BIGINT
            )
    """)

Executed
Executed


In [5]:
v1=253
v2=260
for year in (2024,):
    for month in range(1, 13):
        trino_exec(f"""
                   insert into conformance_voltwatt 
                    with data as (
                        select site_id, t_stamp, sum(power*circuit_polarity/1000) as P_kW, 
                                avg(voltage) as V, ac_capacity_kw
                    from 
                    (select circuit_id, t_stamp, power, voltage 
                    from ts where year={year} and month={month} and is_pv=True and voltage > 0 and voltage < 300) as ts1
                    inner join meta_up23c as m on ts1.circuit_id = m.circuit_id
                    group by site_id, t_stamp, ac_capacity_kw
                    having avg(voltage) > 253),
                    pq as (
                        select site_id, t_stamp, P_kW, V, ac_capacity_kw,
                    (case when V < {v1} then ac_capacity_kw 
                    when V > {v2} then .2 * ac_capacity_kw
                    else ((ac_capacity_kw - .2 * ac_capacity_kw) / ({v1} - {v2}) * (V - {v2}) + .2 * ac_capacity_kw) end) + 0.04*ac_capacity_kw as max_P_volt_watt
                    from data
                    ),
                    pq2 as (select site_id, t_stamp, day(t_stamp) as day, month(t_stamp) as month, year(t_stamp) as year, P_kW, V, ac_capacity_kw, max_P_volt_watt,
                    greatest(0, P_kW - max_P_volt_watt) as nonconformance_voltwatt 
                    from pq
                    )
                    
                    select year, month, day, site_id, 
                    sum(nonconformance_voltwatt) as nonconformance_voltwatt_sum,
                    sum(case when nonconformance_voltwatt > 0 then 1 else 0 end) as nonconformance_voltwatt_count,
                    count(nonconformance_voltwatt) as total_count
                    from pq2
                    group by year, month, day, site_id
                    order by nonconformance_voltwatt_sum desc
                    """)

Executed
Executed
Executed
Executed
Executed
Executed
Executed
Executed
Executed
Executed
Executed
Executed


In [ ]:
def get_max_P(V, Srated=1, v1=253, v2=260):
    if V is None:
        return None
    if V < v1:
        return Srated
    elif V > v2:
        return .2 * Srated
    else:
        m = (Srated - .2*Srated) / (v1 - v2)
        P = m * (V - v2) + .2*Srated
        return P

In [13]:
v1=253
v2=260
iceberg_sql(f"""
                    with data as (
                        select site_id, t_stamp, sum(power*circuit_polarity/1000) as P_kW, 
                                avg(voltage) as V, ac_capacity_kw
                    from 
                    (select circuit_id, t_stamp, power, voltage 
                    from ts where year=2024 and month=1 and is_pv=True and voltage > 0 and voltage < 300) as ts1
                    inner join meta_up23c as m on ts1.circuit_id = m.circuit_id
                    group by site_id, t_stamp, ac_capacity_kw
                    having avg(voltage) > 253),
                    pq as (
                        select site_id, t_stamp, P_kW, V, ac_capacity_kw,
                    (case when V < {v1} then ac_capacity_kw 
                    when V > {v2} then .2 * ac_capacity_kw
                    else ((ac_capacity_kw - .2 * ac_capacity_kw) / ({v1} - {v2}) * (V - {v2}) + .2 * ac_capacity_kw) end) + 0.04*ac_capacity_kw as max_P_volt_watt
                    from data
                    ),
                    pq2 as (select site_id, t_stamp, day(t_stamp) as day, month(t_stamp) as month, year(t_stamp) as year, P_kW, V, ac_capacity_kw, max_P_volt_watt,
                    greatest(0, P_kW - max_P_volt_watt) as nonconformance_voltwatt 
                    from pq
                    )
                    
                    select site_id, day, month, year, 
                    sum(nonconformance_voltwatt) as nonconformance_voltwatt_sum,
                    sum(case when nonconformance_voltwatt > 0 then 1 else 0 end) as nonconformance_voltwatt_count,
                    count(nonconformance_voltwatt) as total_count
                    from pq2
                    group by site_id, day, month, year
                    order by nonconformance_voltwatt_sum desc
                    limit 10""")

,site_id,day,month,year,nonconformance_voltwatt_sum,nonconformance_voltwatt_count,total_count
0,1124511829,19,1,2024,544.442182,58,89
1,465008538,18,1,2024,435.721716,92,180
2,465008538,19,1,2024,405.846543,79,146
3,755163749,22,1,2024,394.168985,68,84
4,1124511829,13,1,2024,370.185861,56,92
5,755163749,11,1,2024,352.225633,71,93
6,465008538,11,1,2024,329.451779,77,229
7,814602644,4,1,2024,327.273590,63,77
8,814602644,3,1,2024,314.986680,64,74
9,755163749,31,1,2024,314.239765,56,62


In [64]:
iceberg_sql("select circuit_id, circuit_polarity from meta_up23c where circuit_id = 467634")

,circuit_id,circuit_polarity


In [65]:
iceberg_sql("select * from meta_up23c limit 2")

,site_id,state,postcode,longitude,latitude,dnsp_name,dc_capacity_kw,ac_capacity_kw,export_limit_kw,monitoring_start,...,circuit_type,is_pv,min_time,max_time,v_95,v_05,m_id,n_long,n_lat,distance_km
0,1944472430,NSW,2502,150.85,-34.47,Endeavour,5.18,5.0,3.0,2021-08-09,...,pv_site_net,True,2024-07-23 04:45:00,2024-07-24 07:05:00,244.3,238.65,M43,150.84,-34.46,1.569777
1,1944472430,NSW,2502,150.85,-34.47,Endeavour,5.18,5.0,3.0,2021-08-09,...,ac_load_net,False,2024-07-23 04:45:00,2024-07-24 07:05:00,244.3,238.65,M43,150.84,-34.46,1.569777


In [18]:
iceberg_sql("""
            with data as (
                select ts.circuit_id, t_stamp, power as P_kW, 
                       energy_reactive*circuit_polarity/1000*12 as Q_kvar, voltage as V
            from ts left join meta_up23c as m on ts.circuit_id = m.circuit_id
            where year=2024 and month=1 and ts.is_pv=True and ts.circuit_id = 467634)
            select * from data order by t_stamp limit 10
            """)

,circuit_id,t_stamp,P_kW,Q_kvar,V
0,467634,2024-01-01 00:00:00,4473.9233,None,241.45
1,467634,2024-01-01 00:05:00,4523.3200,None,241.25
2,467634,2024-01-01 00:10:00,4535.8167,None,241.65
3,467634,2024-01-01 00:15:00,4589.9300,None,242.35
4,467634,2024-01-01 00:20:00,4619.6267,None,241.65
5,467634,2024-01-01 00:25:00,4716.0133,None,241.90
6,467634,2024-01-01 00:30:00,4767.7167,None,242.15
7,467634,2024-01-01 00:35:00,4758.5533,None,242.00
8,467634,2024-01-01 00:40:00,4803.2267,None,242.10
9,467634,2024-01-01 00:45:00,4833.1000,None,243.00
